# Task 2

In [1]:
import nltk
from nltk import CFG
from nltk.parse import EarleyChartParser

In [2]:
grammar_text = r"""
S -> NP VP
NP -> Det ADJNP | ADJNP
ADJNP -> ADJ NOUN_ENUM | NOUN_ENUM
NOUN_ENUM -> N | N CONJ N
VP -> PRED | PRED COMPL_ENUM
PRED -> TRANZ_PRED | INTRANZ_PRED
TRANZ_PRED -> 'interviewed' Det ADJNP
INTRANZ_PRED -> 'can' V_INF | 'were' 'waiting'
V_INF -> 'be' ADJ 
COMPL_ENUM -> COMPL COMPL_ENUM | COMPL
COMPL -> ADV | PP
PP -> P NP

Det -> 'the'
CONJ -> 'and'
P -> 'with' | 'near'
N -> 'professors' | 'men' | 'women' | 'reporter' | 'actor' | 'camera' | 'entrance'
ADV -> 'outside'
ADJ -> 'visiting' | 'annoying' | 'old'
"""

In [3]:
sentences = [
    "Visiting professors can be annoying",
    "Old men and women were waiting outside",
    "The reporter interviewed the actor with the camera near the entrance"
]

grammar = CFG.fromstring(grammar_text)
parser = EarleyChartParser(grammar)

for sentence in sentences: 
    sentence =  sentence.lower()
    tokens = sentence.split()

    parse_trees = list(parser.parse(tokens))

    if parse_trees:
        print(f"Number of parse trees: {len(parse_trees)}")
        for number, tree in enumerate(parse_trees, start=1):
            print(f"\nParse tree {number}:")
            print(tree)
            tree.pretty_print()
    else:
        print("No parse found for the sentence.")


Number of parse trees: 1

Parse tree 1:
(S
  (NP (ADJNP (ADJ visiting) (NOUN_ENUM (N professors))))
  (VP (PRED (INTRANZ_PRED can (V_INF be (ADJ annoying))))))
                           S                             
            _______________|_______                       
           |                       VP                    
           |                       |                      
           NP                     PRED                   
           |                       |                      
         ADJNP                INTRANZ_PRED               
    _______|_______         _______|_________             
   |           NOUN_ENUM   |               V_INF         
   |               |       |        _________|______      
  ADJ              N       |       |               ADJ   
   |               |       |       |                |     
visiting       professors can      be            annoying

Number of parse trees: 1

Parse tree 1:
(S
  (NP (ADJNP (ADJ old) (NOUN_ENUM (N

## Task 3

In [4]:
import spacy

In [5]:
def print_dependency_tree(token, indent=""):
    print(f"{indent}{token.text} ({token.dep_})")
    for child in token.children:
        print_dependency_tree(child, indent + "  +-- ")

In [6]:
nlp = spacy.load("en_core_web_sm")

sentences = [
    "Visiting professors can be annoying",
    "Old men and women were waiting outside",
    "The reporter interviewed the actor with the camera near the entrance",
]

for sentence in sentences:
    doc = nlp(sentence)

    print(f"Sentence: {sentence}")
    print("Token            POS       Head             Dependency")
    print("-" * 65)
    for token in doc:
        print(f"{token.text:<16} {token.pos_:<9} {token.head.text:<16} {token.dep_}")

    print("\nDependency tree:")
    print("-" * 65)
    for token in doc:
        if token.dep_ == "ROOT":
            print_dependency_tree(token)
    print("=" * 80)


Sentence: Visiting professors can be annoying
Token            POS       Head             Dependency
-----------------------------------------------------------------
Visiting         VERB      professors       amod
professors       NOUN      be               nsubj
can              AUX       be               aux
be               AUX       be               ROOT
annoying         ADJ       be               acomp

Dependency tree:
-----------------------------------------------------------------
be (ROOT)
  +-- professors (nsubj)
  +--   +-- Visiting (amod)
  +-- can (aux)
  +-- annoying (acomp)
Sentence: Old men and women were waiting outside
Token            POS       Head             Dependency
-----------------------------------------------------------------
Old              ADJ       men              amod
men              NOUN      waiting          nsubj
and              CCONJ     men              cc
women            NOUN      men              conj
were             AUX       waiting  

In [325]:
cfg_rules = [
    # S rules
    ("S", ["NP", "VP"], 0.8),
    ("S", ["Aux", "NP", "VP"], 0.1),
    ("S", ["VP"], 0.05),
    ("S", ["'the'", "Nominal", "VP"], 0.05),
    
    # NP rules
    ("NP", ["Pronoun"], 0.2),
    ("NP", ["Proper-Noun"], 0.2),
    ("NP", ["Det", "Nominal"], 0.6),
    
    # Nominal rules
    ("Nominal", ["Noun"], 0.3),
    ("Nominal", ["Nominal", "Noun"], 0.2),
    ("Nominal", ["Nominal", "PP"], 0.5),
    
    # VP rules
    ("VP", ["Verb"], 0.2),
    ("VP", ["Verb", "NP"], 0.5),
    ("VP", ["VP", "PP"], 0.3),
    
    # PP rules
    ("PP", ["Prep", "NP"], 1.0),
    
    # Det lexical rules
    ("Det", ["'the'"], 0.6),
    ("Det", ["'a'"], 0.2),
    ("Det", ["'that'"], 0.1),
    ("Det", ["'this'"], 0.1),
    
    # Noun lexical rules
    ("Noun", ["'book'"], 0.1),
    ("Noun", ["'flight'"], 0.5),
    ("Noun", ["'meal'"], 0.2),
    ("Noun", ["'money'"], 0.2),
    
    # Verb lexical rules
    ("Verb", ["'book'"], 0.5),
    ("Verb", ["'include'"], 0.2),
    ("Verb", ["'prefer'"], 0.3),
    
    # Pronoun lexical rules
    ("Pronoun", ["'I'"], 0.5),
    ("Pronoun", ["'he'"], 0.1),
    ("Pronoun", ["'she'"], 0.1),
    ("Pronoun", ["'me'"], 0.3),
    
    # Proper-Noun lexical rules
    ("Proper-Noun", ["'Houston'"], 0.8),
    ("Proper-Noun", ["'NWA'"], 0.2),
    
    # Aux lexical rules
    ("Aux", ["'does'"], 1.0),
    
    # Prep lexical rules
    ("Prep", ["'from'"], 0.25),
    ("Prep", ["'to'"], 0.25),
    ("Prep", ["'on'"], 0.1),
    ("Prep", ["'near'"], 0.2),
    ("Prep", ["'through'"], 0.2),
]

In [326]:
def is_symbol_terminal(symbol: str) -> bool:
    return symbol.startswith("'") and symbol.endswith("'")

In [327]:
def is_rule_in_cnf(rule: tuple[str, list[str], float]) -> bool:
    rhs = rule[1]
    match len(rhs):
        case 1: return is_symbol_terminal(rhs[0])
        case 2: return not is_symbol_terminal(rhs[0]) and not is_symbol_terminal(rhs[1])
        case _: return False

In [328]:
def partition_cnf_rules(rules: list[tuple[str, list[str], float]]) -> tuple[list[tuple[str, list[str], float]], list[tuple[str, list[str], float]]]:
    cnf_rules = []
    non_cnf_rules = []
    for rule in rules:
        if is_rule_in_cnf(rule):
            cnf_rules.append(rule)
        else:
            non_cnf_rules.append(rule)

    return cnf_rules, non_cnf_rules

In [329]:
def search_rules_by_lhs(rules: list[tuple[str, list[str], float]], lhs: str) -> list[tuple[str, list[str], float]]:
    return list(filter(lambda rule: rule[0] == lhs, rules))

# def search_rules_by_rhs(rules: list[tuple[str, list[str], float]], rhs: list[str]) -> list[tuple[str, list[str], float]]:
#     return list(filter(lambda rule: rule[1] == rhs, rules))

In [330]:
def rule_to_string(rule: tuple[str, list[str], float]) -> str:
    lhs, rhs, probability = rule
    result = f"{lhs} -> "
    for symbol in rhs:
        result += f"{symbol} "

    result += f"({probability:.2f})"
    return result

In [331]:
from collections import deque

In [332]:
def eliminate_single_non_terminal(cnf_rules: list[tuple[str, list[str], float]], non_cnf_rules: deque[tuple[str, list[str], float]], rule: tuple[str, list[str], float]):
    lhs, rhs, probability = rule
    replacement_rules = search_rules_by_lhs(cnf_rules, rhs[0])
    if len(replacement_rules) == 0:
        non_cnf_rules.appendleft(rule)
    else:
        for replacement_rule in replacement_rules:
            new_rule = (lhs, replacement_rule[1], probability * replacement_rule[2])
            non_cnf_rules.appendleft(new_rule)
            print(f"[{rule_to_string(rule)}] => [{rule_to_string(new_rule)}]")

In [333]:
def eliminate_terminals(cnf_rules: list[tuple[str, list[str], float]], new_non_terminal_counter: int, rule: tuple[str, list[str], float]):
    _, rhs, _ = rule
    print(f"len >= 2: [{rule_to_string(rule)}]")
    for i, symbol in enumerate(rhs):
        if not is_symbol_terminal(symbol):
            continue

        # replacement_rules = search_rules_by_rhs(cnf_rules, [symbol])

        # if len(replacement_rules) == 0:

        # add new non-terminal
        new_non_terminal = f"X{new_non_terminal_counter}"
        new_non_terminal_counter += 1

        # just in case the non-terminal we're trying to add already exists
        while len(search_rules_by_lhs(cnf_rules, new_non_terminal)) > 0:
            new_non_terminal = f"X{new_non_terminal_counter}"
            new_non_terminal_counter += 1

        rhs[i] = new_non_terminal
        new_rule = (new_non_terminal, [symbol], 1)
                
        cnf_rules.append(new_rule)
        print(f" - [{rule_to_string(new_rule)}]")
        
        # else:
        #     # use existing rule
        #     replacement_rule = replacement_rules[0]
        #     rhs[i] = replacement_rule[0]
        #     print(f" - [{rule_to_string(replacement_rule)}]")

    print(f"Non_term: [{rule_to_string(rule)}]")
            
    return new_non_terminal_counter

In [334]:
def eliminate_non_terminal_pairs(cnf_rules: list[tuple[str, list[str], float]], non_cnf_rules: deque[tuple[str, list[str], float]], new_non_terminal_counter: int, rule: tuple[str, list[str], float]):
    lhs, rhs, probability = rule

    # replacement_rules = search_rules_by_rhs(cnf_rules, rhs[:2])
    # if len(replacement_rules) == 0:

    # add new non-terminal
    new_non_terminal = f"X{new_non_terminal_counter}"
    new_non_terminal_counter += 1

    # just in case the non-terminal we're trying to add already exists
    while len(search_rules_by_lhs(cnf_rules, new_non_terminal)) > 0:
        new_non_terminal = f"X{new_non_terminal_counter}"
        new_non_terminal_counter += 1

    new_rule_1 = (lhs, [new_non_terminal, *rhs[2:]], probability)
    new_rule_2 = (new_non_terminal, rhs[:2], 1)

    non_cnf_rules.appendleft(new_rule_1)
    cnf_rules.append(new_rule_2)
    print(f"[{rule_to_string(rule)}] => [{rule_to_string(new_rule_1)}] + [{rule_to_string(new_rule_2)}]")
    
    # else:
    #     # use existing rule
    #     replacement_rule = replacement_rules[0]
    #     new_rule = (lhs, [replacement_rule[0], *rhs[2:]], probability)
    #     non_cnf_rules.appendleft(new_rule)
    #     print(f"[{rule_to_string(rule)}] => [{rule_to_string(new_rule)}] + [{rule_to_string(replacement_rule)}]")
        
    return new_non_terminal_counter

In [335]:
# def combine_same_rules(rules: list[tuple[str, list[str], float]]) -> list[tuple[str, list[str], float]]:
#     rules = rules.copy()
#     rules.sort(key=lambda rule: (rule[0], rule[1]))
    
#     combined_rules = rules[:1]
#     for rule in rules[1:]:
#         combined_rule = combined_rules.pop()

#         if rule[0] == combined_rule[0] and rule[1] == combined_rule[1]:
#             combined_rule = (rule[0], rule[1], rule[2] + combined_rule[2])
#             combined_rules.append(combined_rule)
#         else:
#             combined_rules.append(combined_rule)
#             combined_rules.append(rule)
        

#     return combined_rules

In [336]:
def convert_rules_to_cnf(rules: list[tuple[str, list[str], float]]) -> list[tuple[str, list[str], float]]:
    cnf_rules, non_cnf_rules = partition_cnf_rules(rules)

    new_non_terminal_counter = 0
    non_cnf_rules.reverse()
    non_cnf_rules = deque(non_cnf_rules)

    while len(non_cnf_rules) > 0:
        rule = non_cnf_rules.pop()

        if is_rule_in_cnf(rule): 
            cnf_rules.append(rule)
            continue

        _, rhs, _ = rule
        if len(rhs) == 1:
            # rhs is a single non terminal
            eliminate_single_non_terminal(cnf_rules, non_cnf_rules, rule)
        else:
            # rhs is a list containing terminals and non-terminals
            new_non_terminal_counter = eliminate_terminals(cnf_rules, new_non_terminal_counter, rule)

            if len(rhs) > 2:
                # rhs is a list containing only non-terminals
                new_non_terminal_counter = eliminate_non_terminal_pairs(cnf_rules, non_cnf_rules, new_non_terminal_counter, rule)

    return cnf_rules #combine_same_rules(cnf_rules)                

In [337]:
cnf_rules = convert_rules_to_cnf(cfg_rules)
print()

is_grammar_cnf = all(map(lambda rule: is_rule_in_cnf(rule), cnf_rules))
cnf_rules.sort(key=lambda rule: (rule[0], rule[1]))
print(f"Is grammar CNF now: {is_grammar_cnf}")
print("---------------CNF RULES---------------")
for rule in cnf_rules:
    print(rule_to_string(rule))

len >= 2: [S -> Aux NP VP (0.10)]
Non_term: [S -> Aux NP VP (0.10)]
[S -> Aux NP VP (0.10)] => [S -> X0 VP (0.10)] + [X0 -> Aux NP (1.00)]
[S -> VP (0.05)] => [S -> Verb NP (0.03)]
[S -> VP (0.05)] => [S -> VP PP (0.01)]
len >= 2: [S -> 'the' Nominal VP (0.05)]
 - [X1 -> 'the' (1.00)]
Non_term: [S -> X1 Nominal VP (0.05)]
[S -> X1 Nominal VP (0.05)] => [S -> X2 VP (0.05)] + [X2 -> X1 Nominal (1.00)]
[NP -> Pronoun (0.20)] => [NP -> 'I' (0.10)]
[NP -> Pronoun (0.20)] => [NP -> 'he' (0.02)]
[NP -> Pronoun (0.20)] => [NP -> 'she' (0.02)]
[NP -> Pronoun (0.20)] => [NP -> 'me' (0.06)]
[NP -> Proper-Noun (0.20)] => [NP -> 'Houston' (0.16)]
[NP -> Proper-Noun (0.20)] => [NP -> 'NWA' (0.04)]
[Nominal -> Noun (0.30)] => [Nominal -> 'book' (0.03)]
[Nominal -> Noun (0.30)] => [Nominal -> 'flight' (0.15)]
[Nominal -> Noun (0.30)] => [Nominal -> 'meal' (0.06)]
[Nominal -> Noun (0.30)] => [Nominal -> 'money' (0.06)]
[VP -> Verb (0.20)] => [VP -> 'book' (0.10)]
[VP -> Verb (0.20)] => [VP -> 'include'

# Task 5

### Constituency Parsing

Constituency Parsing este bun pentru aplicatii care necesita **generarea** de fraze. 
De exemplu, la un moment dat avem o propozitie incompleta in care stim ca dupa ea urmeaza un NOUN PHRASE (din regulile de constructie). Cand folosim Constituency, noi avem acces la regulile de constructie a fiecarei parti dintr-o fraza, ceea ce inseamna ca ne va fi usor sa construim un NOUN PHRASE potrivit pentru propozitia incompleta.

### Dependency Parsing

Dependency Parsing este util pentru **analiza semantica** a unei fraze. De exemplu, putem sti care este actiunea descrisa, de catre cine a fost facuta, cum/in ce mod este facuta, asupra cui este facuta s.a.m.d. Analiza este usor de facut in urma unui dependency parsing pentru ca acest tip de parsare urmareste relatiile dintre cuvinte si felul in care acestea se influenteaza reciproc.
